# 03 – Thử nghiệm và Phân tích Độ nhạy CLV (Customer Lifetime Value)

Notebook này thực hiện các phân tích theo spec `docs/05-clv-methodology.md`:
1. Tính toán CLV tiềm năng, CLV kỳ vọng và Value at Risk (VaR)
2. Đánh giá tương quan Spearman với `Lifetime_Value` lịch sử
3. Phân tích độ nhạy của lãi suất chiết khấu $d$ ($8\%, 10\%, 12\%$)
4. Xuất ngưỡng phân vị $P_{33}$ và $P_{67}$ cho `clv_config.json`

In [1]:
import sys
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# Thêm root vào path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ml_engine.src.clv import clv_components, build_clv_config, sensitivity_analysis_d


## 1. Nạp dữ liệu và Model Churn đã Calibrate

In [2]:
raw_data_path = root / 'ml_engine' / 'data' / 'raw' / 'ecommerce_customer_dataset.csv'
artifact_path = root / 'ml_engine' / 'artifacts' / 'churn_v1.joblib'

df = pd.read_csv(raw_data_path)
artifact = joblib.load(artifact_path)
model = artifact['model']
tau = artifact.get('tau', 0.186)

print(f'Số lượng khách hàng: {len(df):,}')
print(f'Model thuật toán: {artifact.get("algo", "lightgbm")}, Ngưỡng tau: {tau:.4f}')

p_churn = model.predict_proba(df)[:, 1]
print(f'Tỷ lệ churn dự báo trung bình: {p_churn.mean():.4f}')


Số lượng khách hàng: 50,000
Model thuật toán: lightgbm, Ngưỡng tau: 0.1857


Tỷ lệ churn dự báo trung bình: 0.2954


## 2. Tính toán các thành phần CLV

In [3]:
# Tính CLV kỳ vọng, tiềm năng và VaR
clv_df = clv_components(df, p_churn, g=0.33, d=0.10, horizon=3, freq_cap=41.0)
df_scored = pd.concat([df, clv_df], axis=1)

print('=== THỐNG KÊ MÔ TẢ CLV (USD) ===')
display_stats = clv_df.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
print(display_stats)

corr, _ = spearmanr(clv_df['clv_potential'], df['Lifetime_Value'])
print(f'\nSpearman correlation giữa CLV tiềm năng và Lifetime_Value: {corr:.4f} (Tiêu chí >= 0.5: ĐẠT)')


=== THỐNG KÊ MÔ TẢ CLV (USD) ===
                     mean          std  min       25%     50%       75%  \
clv_pred       451.148383  1338.392491  0.0   64.6875  234.56  541.4850   
clv_potential  367.175907   899.819051  0.0  112.1475  216.39  432.4325   
value_at_risk  106.379539   458.911384  0.0    9.8100   24.17   78.7325   

                     max  
clv_pred       225088.17  
clv_potential  132834.24  
value_at_risk   53845.07  

Spearman correlation giữa CLV tiềm năng và Lifetime_Value: 0.6144 (Tiêu chí >= 0.5: ĐẠT)


## 3. Phân tích độ nhạy của tỷ lệ chiết khấu d

In [4]:
sens_df = sensitivity_analysis_d(df, p_churn, d_values=(0.08, 0.10, 0.12), freq_cap=41.0)
print('=== BẢNG PHÂN TÍCH ĐỘ NHẠY LÃI SUẤT CHIẾT KHẤU d ===')
print(sens_df.to_string(index=False))


=== BẢNG PHÂN TÍCH ĐỘ NHẠY LÃI SUẤT CHIẾT KHẤU d ===
   d  mean_clv_potential  median_clv_potential  mean_clv_expected  median_clv_expected    P33    P67  spearman_vs_base_potential  spearman_vs_base_expected  tier_swap_count  tier_swap_pct
0.08              378.94                223.32             466.98               242.69 147.05 345.80                         1.0                        1.0                0            0.0
0.10              367.18                216.39             451.15               234.56 142.49 335.07                         1.0                        1.0                0            0.0
0.12              356.05                209.84             436.22               226.92 138.17 324.92                         1.0                        1.0                0            0.0


## 4. Ngưỡng phân vị và Cấu hình clv_config.json

In [5]:
# clv_config.json do python -m ml_engine.src.train sinh ra; notebook chỉ đọc
config = json.loads((root / 'ml_engine' / 'artifacts' / 'clv_config.json').read_text(encoding='utf-8'))
print(json.dumps(config, indent=2))


{
  "g": 0.33,
  "d": 0.1,
  "horizon": 3,
  "r_bar": 0.7046,
  "freq_cap": 41.0,
  "P33": 142.49,
  "P67": 335.07
}
